### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [52]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [53]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [54]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [55]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [56]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [57]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [58]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [59]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [60]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [61]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [62]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [63]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [64]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [65]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [66]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [67]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [68]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 1911, 1825, 1828], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [69]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [70]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [71]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [72]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [73]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [74]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


## 1. Vectorización 

In [75]:
np.random.seed(42)
doc_indices = np.random.choice(len(newsgroups_train.data), 5, replace=False)

for doc_idx in doc_indices:
    label = newsgroups_train.target_names[y_train[doc_idx]]
    print(f"\n{'─' * 65}")
    print(f"Doc #{doc_idx:5d} | Categoria: {label}")
    print(f"{'─' * 65}")
    text_preview = ' '.join(newsgroups_train.data[doc_idx].split()[:30])
    print(f"Extracto: {text_preview} ...")

    sim = cosine_similarity(X_train[doc_idx], X_train)[0]
    sorted_idx = np.argsort(sim)[::-1]
    top5 = [i for i in sorted_idx if i != doc_idx][:5]

    print(f"\n  5 documentos mas similares:")
    same_class = 0
    for rank, idx in enumerate(top5, 1):
        cat = newsgroups_train.target_names[y_train[idx]]
        match = "[OK]" if y_train[idx] == y_train[doc_idx] else "[--]"
        print(f"    {rank}. #{idx:5d} | sim={sim[idx]:.4f} | {match} {cat}")
        if y_train[idx] == y_train[doc_idx]:
            same_class += 1
    print(f"\n  Coincidencia de categoria: {same_class}/5")


─────────────────────────────────────────────────────────────────
Doc # 7492 | Categoria: comp.sys.mac.hardware
─────────────────────────────────────────────────────────────────
Extracto: Could someone please post any info on these systems. Thanks. BoB -- ---------------------------------------------------------------------- Robert Novitskey | "Pursuing women is similar to banging one's head rrn@po.cwru.edu | against a wall...with less ...

  5 documentos mas similares:
    1. #10935 | sim=0.6665 | [OK] comp.sys.mac.hardware
    2. # 7258 | sim=0.3476 | [--] comp.sys.ibm.pc.hardware
    3. # 4971 | sim=0.1799 | [OK] comp.sys.mac.hardware
    4. # 4303 | sim=0.1547 | [--] misc.forsale
    5. #  645 | sim=0.1414 | [OK] comp.sys.mac.hardware

  Coincidencia de categoria: 3/5

─────────────────────────────────────────────────────────────────
Doc # 3546 | Categoria: comp.os.ms-windows.misc
─────────────────────────────────────────────────────────────────
Extracto: Don't bother if you have 

### Interpretación

Los resultados revelan que la efectividad de TF-IDF depende fuertemente de cuán específico es el vocabulario del documento:

**Casos exitosos (3/5):**
- Doc #7492 (`comp.sys.mac.hardware`): vocabulario técnico específico de hardware Apple → similitud alta (0.67) con documentos de la misma categoría.
- Doc #5582 (`misc.forsale`): anuncio de venta de componentes → los similares también son ventas o publicaciones sobre hardware, semánticamente coherente aunque algunas etiquetas difieran.

**Caso fallido (0/5):**
- Doc #3546 (`comp.os.ms-windows.misc`): habla de software de backup en PC. Su vocabulario (CPBackup, formatos, compresión) es indistinguible del que usa `comp.sys.ibm.pc.hardware`. TF-IDF no puede separar estas categorías porque **comparten vocabulario** — el límite semántico entre software de Windows y hardware IBM es difuso.

**Caso parcial (2/5):**
- Doc #4793 (`talk.politics.guns`): trata regulaciones de armas en parques nacionales, con lenguaje político general. Aparecen documentos de `talk.politics.misc`, `sci.crypt` y `alt.atheism` porque el texto no tiene vocabulario de armas lo suficientemente específico para separarlo del debate político general.

**Conclusión**: TF-IDF funciona bien cuando los documentos tienen términos especializados propios de su categoría. Falla cuando el vocabulario es ambiguo, el texto es corto/genérico, o las categorías son temáticamente adyacentes (como los grupos `comp.*` entre sí).

## 2. Clasificación por Prototipos (Zero-Shot)

Para cada documento de test, se calcula su similitud coseno con **todos** los documentos de entrenamiento y se le asigna la clase del documento de entrenamiento más similar. No se entrena ningún modelo explícito: la "memoria" del clasificador son los documentos de train.

In [76]:
print("Clasificando documentos de test por similitud con train...")
print(f"  Test: {X_test.shape[0]} docs | Train: {X_train.shape[0]} docs\n")

# Procesamos en lotes para no agotar memoria RAM
batch_size = 300
y_pred_proto = []

for start in range(0, X_test.shape[0], batch_size):
    end = min(start + batch_size, X_test.shape[0])
    sim_batch = cosine_similarity(X_test[start:end], X_train)   # (batch, n_train)
    best_idx = np.argmax(sim_batch, axis=1)                     # doc de train más similar
    y_pred_proto.extend(y_train[best_idx])
    if start % 1500 == 0:
        print(f"  Procesados {end}/{X_test.shape[0]} documentos...")

y_pred_proto = np.array(y_pred_proto)

f1_proto = f1_score(y_test, y_pred_proto, average='macro')
print(f"\nF1-Score Macro (clasificacion por prototipos): {f1_proto:.4f}")

Clasificando documentos de test por similitud con train...
  Test: 7532 docs | Train: 11314 docs

  Procesados 300/7532 documentos...
  Procesados 1800/7532 documentos...
  Procesados 3300/7532 documentos...
  Procesados 4800/7532 documentos...
  Procesados 6300/7532 documentos...
  Procesados 7532/7532 documentos...

F1-Score Macro (clasificacion por prototipos): 0.5050


### Interpretación

El clasificador por prototipos obtuvo un **F1 Macro de 0.5050** sobre 20 clases, sin entrenar ningún modelo explícito.

**¿Es un buen resultado?**
- Un clasificador aleatorio sobre 20 clases obtendría ~0.05. Llegar a 0.50 solo buscando el vecino más similar es significativo.
- El modelo Naïve Bayes baseline (siguiente sección) supera este valor, lo que tiene sentido: NB aprende distribuciones globales de términos por clase, mientras que este enfoque solo se apoya en el documento de train más parecido — si ese documento es ruidoso o atípico, la predicción falla.

**Características del enfoque:**
- **Sin entrenamiento**: no aprende parámetros, memoriza los ejemplos de train tal cual.
- **Inferencia costosa**: para cada documento de test se compara contra los 11.314 de train → O(n_test × n_train). Por eso se procesó en lotes.
- **Sensible a outliers**: si el documento más similar de train es un ejemplo raro o mal etiquetado, la predicción es incorrecta aunque el resto de la clase sea fácilmente separable.
- **Relacionado con k-NN**: es esencialmente un 1-NN con distancia coseno sobre vectores TF-IDF.

## 3. Optimización del Modelo Naïve Bayes

Se exploran combinaciones de:
- **Vectorizador**: `TfidfVectorizer` vs `CountVectorizer`, con variaciones en `stop_words`, `min_df`, `max_df` y `sublinear_tf`.
- **Modelo**: `MultinomialNB` vs `ComplementNB`, con distintos valores de `alpha` (suavizado de Laplace).



In [77]:
experiments = [
    # (descripcion, vectorizador, modelo)
    (
        "Baseline (TF-IDF default + MultinomialNB)",
        TfidfVectorizer(),
        MultinomialNB()
    ),
    (
        "TF-IDF stop_words + MultinomialNB",
        TfidfVectorizer(stop_words='english'),
        MultinomialNB()
    ),
    (
        "TF-IDF stop_words + min_df=2 + MultinomialNB",
        TfidfVectorizer(stop_words='english', min_df=2),
        MultinomialNB()
    ),
    (
        "TF-IDF stop_words + min_df=2 + sublinear_tf + MultinomialNB",
        TfidfVectorizer(stop_words='english', min_df=2, sublinear_tf=True),
        MultinomialNB()
    ),
    (
        "TF-IDF stop_words + min_df=2 + ComplementNB",
        TfidfVectorizer(stop_words='english', min_df=2),
        ComplementNB()
    ),
    (
        "TF-IDF stop_words + min_df=2 + sublinear_tf + ComplementNB",
        TfidfVectorizer(stop_words='english', min_df=2, sublinear_tf=True),
        ComplementNB()
    ),
    (
        "TF-IDF stop_words + min_df=3 + max_df=0.8 + sublinear_tf + ComplementNB",
        TfidfVectorizer(stop_words='english', min_df=3, max_df=0.8, sublinear_tf=True),
        ComplementNB()
    ),
    (
        "CountVectorizer stop_words + min_df=2 + MultinomialNB",
        CountVectorizer(stop_words='english', min_df=2),
        MultinomialNB()
    ),
    (
        "CountVectorizer stop_words + min_df=2 + ComplementNB",
        CountVectorizer(stop_words='english', min_df=2),
        ComplementNB()
    ),
    (
        "TF-IDF stop_words + min_df=2 + sublinear_tf + ComplementNB alpha=0.1",
        TfidfVectorizer(stop_words='english', min_df=2, sublinear_tf=True),
        ComplementNB(alpha=0.1)
    ),
    (
        "TF-IDF stop_words + min_df=2 + sublinear_tf + ComplementNB alpha=0.5",
        TfidfVectorizer(stop_words='english', min_df=2, sublinear_tf=True),
        ComplementNB(alpha=0.5)
    ),
]

print(f"{'#':>2}  {'F1 Macro':>9}  {'Vocab':>7}  Descripcion")
print("─" * 85)

results = []
for i, (desc, vect, model) in enumerate(experiments, 1):
    Xtr = vect.fit_transform(newsgroups_train.data)
    Xte = vect.transform(newsgroups_test.data)
    model.fit(Xtr, y_train)
    y_pred_exp = model.predict(Xte)
    f1 = f1_score(y_test, y_pred_exp, average='macro')
    vocab_size = len(vect.vocabulary_)
    results.append((f1, desc))
    print(f"{i:>2}  {f1:>9.4f}  {vocab_size:>7,}  {desc}")

best_f1, best_desc = max(results)
print(f"\n  Mejor configuracion: {best_desc}")
print(f"  Mejor F1 Macro: {best_f1:.4f}")

 #   F1 Macro    Vocab  Descripcion
─────────────────────────────────────────────────────────────────────────────────────
 1     0.5854  101,631  Baseline (TF-IDF default + MultinomialNB)
 2     0.6468  101,322  TF-IDF stop_words + MultinomialNB
 3     0.6512   39,115  TF-IDF stop_words + min_df=2 + MultinomialNB
 4     0.6437   39,115  TF-IDF stop_words + min_df=2 + sublinear_tf + MultinomialNB
 5     0.6943   39,115  TF-IDF stop_words + min_df=2 + ComplementNB
 6     0.6921   39,115  TF-IDF stop_words + min_df=2 + sublinear_tf + ComplementNB
 7     0.6907   26,747  TF-IDF stop_words + min_df=3 + max_df=0.8 + sublinear_tf + ComplementNB
 8     0.6179   39,115  CountVectorizer stop_words + min_df=2 + MultinomialNB
 9     0.6408   39,115  CountVectorizer stop_words + min_df=2 + ComplementNB
10     0.6878   39,115  TF-IDF stop_words + min_df=2 + sublinear_tf + ComplementNB alpha=0.1
11     0.6951   39,115  TF-IDF stop_words + min_df=2 + sublinear_tf + ComplementNB alpha=0.5

  Mejor conf

### Interpretación

La mejor configuración fue **TF-IDF + stop_words + min_df=2 + sublinear_tf + ComplementNB α=0.5** con un F1 Macro de **0.6951**, mejorando el baseline en +0.11.

**Análisis por factor:**

| Factor | Efecto observado |
|--------|-----------------|
| `stop_words='english'` | El mayor salto individual: +0.06 (de 0.5854 a 0.6468). Eliminar palabras funcionales reduce ruido y concentra el vocabulario en términos discriminativos. |
| `min_df=2` | Mejora leve (+0.004). Reduce el vocabulario de 101k a 39k términos descartando hapax legomena (palabras que aparecen en un solo documento y no generalizan). |
| `sublinear_tf` | Perjudica a MultinomialNB (exp. 4 < exp. 3: 0.6437 vs 0.6512) pero es neutro para ComplementNB. MultinomialNB asume que las frecuencias son proporcionales a la probabilidad; aplicar log rompe ese supuesto. |
| MultinomialNB → ComplementNB | Ganancia de +0.043 (de 0.6512 a 0.6943). ComplementNB estima la probabilidad de que un doc pertenezca al **complemento** de cada clase, siendo más robusto cuando las clases tienen desbalance o vocabulario compartido. |
| `max_df=0.8` | Reduce el vocabulario a 26k pero baja levemente el F1 (0.6907). Las palabras muy frecuentes ya tienen peso bajo en TF-IDF, así que filtrarlas con max_df aporta poco y puede eliminar términos útiles. |
| CountVectorizer | Peor que TF-IDF en todos los casos. Las frecuencias brutas sin normalización IDF favorecen términos comunes que no discriminan entre clases. |
| `alpha=0.5` | Mejor que alpha=0.1 y que el default (1.0): F1=0.6951. Un suavizado moderado evita tanto el sobreajuste (α muy bajo) como la dilución de las estimaciones (α muy alto). |

**Conclusión**: las ganancias provienen en orden de importancia de (1) eliminar stop words, (2) usar ComplementNB, y (3) ajustar el suavizado α. El clasificador por prototipos de la parte 2 (F1=0.505) queda superado por el mejor NB (F1=0.695).

## 4. Similaridad entre Palabras (Matriz Término-Documento)

Al **transponer** la matriz documento-término obtenemos una matriz término-documento donde cada fila es un vector que representa a una palabra en el espacio de documentos. Palabras que co-ocurren en los mismos documentos tendrán vectores similares, capturando relaciones semánticas de forma distribucional.

Las 5 palabras elegidas pertenecen a dominios temáticos bien diferenciados del corpus 20newsgroups para obtener resultados interpretables.

In [78]:
# Transponer: de (n_docs x n_terms) a (n_terms x n_docs)
X_term_doc = X_train.T

print(f"Matriz documento-termino:  {X_train.shape}  (docs x terminos)")
print(f"Matriz termino-documento:  {X_term_doc.shape}  (terminos x docs)")

# Palabras elegidas manualmente de dominios tematicos distintos:
#   'space'    -> sci.space
#   'gun'      -> talk.politics.guns
#   'god'      -> soc.religion.christian / talk.religion.misc
#   'hockey'   -> rec.sport.hockey
#   'computer' -> comp.*
seed_words = ['space', 'gun', 'god', 'hockey', 'computer']

# Verificar que esten en el vocabulario
seed_words = [w for w in seed_words if w in tfidfvect.vocabulary_]
print(f"\nPalabras de analisis: {seed_words}\n")

for word in seed_words:
    word_idx = tfidfvect.vocabulary_[word]
    word_vec = X_term_doc[word_idx]                          # vector de la palabra

    # Similitud coseno contra todos los terminos
    sim_words = cosine_similarity(word_vec, X_term_doc)[0]

    # Top 5 (excluimos la palabra misma)
    sorted_w = np.argsort(sim_words)[::-1]
    top5_words = [idx2word[i] for i in sorted_w if idx2word[i] != word][:5]

    print(f"  '{word}'  ->  mas similares: {top5_words}")

Matriz documento-termino:  (11314, 101631)  (docs x terminos)
Matriz termino-documento:  (101631, 11314)  (terminos x docs)

Palabras de analisis: ['space', 'gun', 'god', 'hockey', 'computer']

  'space'  ->  mas similares: ['nasa', 'seds', 'shuttle', 'enfant', 'seti']
  'gun'  ->  mas similares: ['guns', 'crime', 'handgun', 'homicides', 'firearms']
  'god'  ->  mas similares: ['jesus', 'bible', 'that', 'existence', 'christ']
  'hockey'  ->  mas similares: ['ncaa', 'nhl', 'affiliates', 'xenophobes', 'sportschannel']
  'computer'  ->  mas similares: ['decwriter', 'deluged', 'harkens', 'shopper', 'the']


### Interpretación

Los resultados varían bastante según cuán específico es el vocabulario de cada palabra:

**Resultados limpios:**

- **`gun`** → `guns, crime, handgun, homicides, firearms`: el resultado más coherente. `talk.politics.guns` tiene vocabulario muy específico y concentrado; todos los similares son semánticamente directos.
- **`space`** → `nasa, seds, shuttle, seti` (+ `enfant`): 4 de 5 son claramente del dominio espacial (NASA, SEDS = Students for the Exploration and Development of Space, SETI). `enfant` es ruido — probablemente aparece en algún documento de `sci.space` por alguna cita o firma, y quedó asociado por co-ocurrencia puntual.
- **`god`** → `jesus, bible, existence, christ` (+ `that`): 4 de 5 coherentes con debates religiosos/filosóficos. `that` es una palabra función que debería tener peso IDF bajo, pero sin `stop_words` en el vectorizador base sigue presente y puede aparecer como ruido.

**Resultados ruidosos:**

- **`hockey`** → `nhl` ✓, `ncaa` ✓ (deportes universitarios), pero `affiliates`, `xenophobes`, `sportschannel` son términos muy raros que co-ocurrieron en algún post específico de hockey y quedaron artificialmente similares por su baja frecuencia global.
- **`computer`** → el peor resultado. `decwriter` (impresora DEC) y `shopper` (revista *Computer Shopper*) tienen relación, pero `deluged`, `harkens` y `the` son completamente no interpretables. El problema: `computer` es una palabra **genérica** que aparece en muchísimos documentos de categorías distintas (`comp.*`, `sci.*`, `misc.*`), lo que hace que su vector en el espacio de documentos sea muy difuso y su similitud con otros términos sea ruidosa.

**Limitación estructural del método:**

Esta representación captura co-ocurrencia a **nivel de documento**, no de ventana de palabras. Dos términos son similares si aparecen en los mismos documentos, independientemente de si aparecen cerca entre sí. Palabras genéricas (como `computer` o `the`) aparecen en documentos muy diversos → su vector es difuso → sus vecinos son aleatorios. Palabras especializadas (como `gun` o `space`) aparecen en documentos concentrados en un tema → vecinos coherentes.

Esto contrasta con embeddings como Word2Vec o GloVe, que usan ventanas de contexto local y producen representaciones mucho más limpias incluso para palabras de uso general.